### Native Bayes

In [ ]:
import pandas as pd
import time
import warnings
import tracemalloc
warnings.filterwarnings('ignore')

from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import f1_score
from scipy.stats import loguniform

In [6]:
def native_bayes(data):
    y = data['collision']
    x = data.drop('collision', axis=1)
    
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=81)

    nb = GaussianNB()

    tracemalloc.start()
    start_time = time.time()

    nb.fit(x_train, y_train)

    training_time = time.time() - start_time
    _, peak_memory = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    y_pred = nb.predict(x_test)
    
    f1_metric = f1_score(y_test, y_pred)
    peak_memory_mb = peak_memory / (1024 * 1024)

    return f1_metric, training_time, peak_memory_mb

#### Mетрики без подбора гиперпараметров

In [ ]:
n_samples = [100, 500, 1000, 3000]
m_features = [5, 8, 11]

results = []

for n in n_samples:
    for m in m_features:
        data = pd.read_csv(f"f1_data/f1_data_{n}_s_{m}_f.csv")
        f1, real_time, mem_usage = nat(data)
        results.append({
            'samples (n)' : n,
            'features (m)' : m,
            'f1-score' : f1,
            'time (sec)' : real_time,
            'memory (MB)': mem_usage
        })

results_df = pd.DataFrame(results)

f1_avg = results_df['f1-score'].mean()
real_time_avg = results_df['time (sec)'].mean()
mem_usage_avg = results_df['memory (MB)'].mean()

print(f'Средняя f1-мера = {round(f1_avg, 3)}')
print(f'Среднее время = {round(real_time_avg, 5)} сек')
print(f'Среднее потребление памяти = {round(mem_usage_avg, 3)} MB')

results_df

Средняя f1-мера = 0.938
Среднее время = 0.00186 сек
Среднее потребление памяти = 0.207 MB


,samples (n),features (m),f1-score,time (sec),memory (MB)
0,100,5,1.000000,0.004809,0.025414
1,100,8,0.967742,0.001923,0.020508
2,100,11,0.857143,0.001733,0.038681
3,500,5,0.974026,0.001638,0.067113
4,500,8,0.935484,0.001509,0.099373
5,500,11,0.884848,0.001856,0.163002
6,1000,5,0.992754,0.001487,0.129471
7,1000,8,0.967509,0.001597,0.193550
8,1000,11,0.868421,0.001400,0.279633
9,3000,5,0.984281,0.001401,0.308289


Наивный Байесовский классификатор показывает `f1-score` больший, чем в Linear SVC, но требует немного больше памяти и меньше времени.

#### Mетрики с подбором гиперпараметров

In [9]:
def native_bayes_params(data):
    y = data['collision']
    x = data.drop('collision', axis=1)
    
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=81)

    param_dist = {
        'var_smoothing': loguniform(1e-10, 1e-7) 
    }       # сглаживание

    nb = GaussianNB()

    random_search = RandomizedSearchCV(
        nb, param_dist, n_iter=10, cv=5, scoring='f1', random_state=81
    )

    tracemalloc.start()
    start_time = time.time()

    random_search.fit(x_train, y_train)

    training_time = time.time() - start_time
    _, peak_memory = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    best_model = random_search.best_estimator_
    y_pred = best_model.predict(x_test)
    
    f1_metric = f1_score(y_test, y_pred)
    best_params = random_search.best_params_
    peak_memory_mb = peak_memory / (1024 * 1024)

    return f1_metric, training_time, peak_memory_mb, best_params

In [11]:
results = []

for n in n_samples:
    for m in m_features:
        data = pd.read_csv(f"f1_data/f1_data_{n}_s_{m}_f.csv")
        f1, real_time, mem_usage, best_p = native_bayes_params(data)
        results.append({
            'samples (n)' : n,
            'features (m)' : m,
            'f1-score' : f1,
            'time (sec)' : real_time,
            'memory (MB)': mem_usage,
            'var_smoothing': best_p['var_smoothing']
        })

results_df = pd.DataFrame(results)

f1_avg = results_df['f1-score'].mean()
real_time_avg = results_df['time (sec)'].mean()
mem_usage_avg = results_df['memory (MB)'].mean()

print(f'Средняя f1-мера = {round(f1_avg, 3)}')
print(f'Среднее время = {round(real_time_avg, 5)} сек')
print(f'Среднее потребление памяти = {round(mem_usage_avg, 3)} MB')

results_df

Средняя f1-мера = 0.938
Среднее время = 0.23289 сек
Среднее потребление памяти = 0.386 MB


,samples (n),features (m),f1-score,time (sec),memory (MB),var_smoothing
0,100,5,1.000000,0.259441,0.107309,1.164816e-09
1,100,8,0.967742,0.227209,0.080677,1.164816e-09
2,100,11,0.857143,0.226205,0.102565,1.164816e-09
3,500,5,0.974026,0.220999,0.166609,1.164816e-09
4,500,8,0.935484,0.244211,0.194026,1.164816e-09
5,500,11,0.884848,0.240408,0.279114,1.164816e-09
6,1000,5,0.992754,0.224424,0.261711,1.164816e-09
7,1000,8,0.967509,0.227851,0.333112,1.164816e-09
8,1000,11,0.868421,0.231161,0.484158,1.164816e-09
9,3000,5,0.984281,0.227805,0.623769,1.164816e-09


После подбора гиперпараметра `f1-score` не изменился. 

Параметр `var_smoothing` отвечает за вычислительную стабильность модели. Он добавляет небольшое значение к дисперсии признаков, чтобы расширить границы нормального распределения. Это предотвращает ситуацию, когда модель приписывает нулевую вероятность редким событиям.